🎯ขั้นตอนที่ 1: กำหนดข้อมูลและการตั้งค่าพื้นฐาน

ก่อนจะเขียนฟังก์ชันใด ๆ คุณจำเป็นต้องมีข้อมูลที่ถูกต้องของเครื่อง Enigma ก่อน
1.1. รวบรวมข้อมูลการเดินสาย (Wiring Data)

คุณต้องกำหนดข้อมูลการเดินสาย (Mapping) ของส่วนประกอบหลัก:

    Rotors: กำหนดการเดินสายของโรเตอร์แต่ละตัว (I, II, III, IV, V) และตำแหน่ง Notch ที่ใช้ในการหมุนโรเตอร์ถัดไป

    Reflectors: กำหนดการเดินสายของแผ่นสะท้อน (B, C)

    Alphabet: กำหนดตัวอักษรมาตรฐาน A−Z เพื่อใช้อ้างอิงการแปลงระหว่างตัวอักษรและดัชนี (0-25)

    เคล็ดลับ: จัดเก็บข้อมูลเหล่านี้ในโครงสร้างที่เข้าถึงง่าย เช่น Dictionary ภายในคลาส EnigmaMachine หรือในโมดูลแยกต่างหาก

⚙️ ขั้นตอนที่ 2: สร้างคลาสส่วนประกอบเดี่ยว

มุ่งเน้นการสร้างคลาสย่อยที่ทำงานได้อย่างสมบูรณ์ในตัวเองก่อน
2.1. คลาส Reflector 🪞

    นี่คือคลาสที่ง่ายที่สุด เพียงแค่ทำการแมปตัวอักษรขาเข้าไปยังตัวอักษรขาออกแบบคงที่

    Method หลัก: reflect(index): รับดัชนี (0-25) และคืนค่าดัชนีที่ถูกสะท้อนกลับ

2.2. คลาส Plugboard 🔌

    ใช้ Dictionary หรือ List เพื่อจัดเก็บการสลับคู่ตัวอักษร เช่น A↔P,Z↔Y

    Method หลัก: swap(char): รับตัวอักษรและคืนค่าตัวอักษรที่ถูกสลับ (หรือตัวอักษรเดิมหากไม่มีการสลับ)

2.3. คลาส Rotor 🌀 (ซับซ้อนที่สุด)

นี่คือหัวใจสำคัญที่คุณต้องใช้เวลา:

    Attributes: ต้องเก็บ wiring, notch, position (ตำแหน่งปัจจุบัน), และ ring_setting (การตั้งค่าวงแหวน)

    Method หลัก 1: forward(index): เข้ารหัสสัญญาณ จากขวาไปซ้าย (ขาเข้า) ต้องคำนวณการเลื่อนเนื่องจาก position และ ring_setting

    Method หลัก 2: backward(index): เข้ารหัสสัญญาณ จากซ้ายไปขวา (ขาออก) เป็นกระบวนการย้อนกลับของ forward

    Method หลัก 3: rotate(): หมุนโรเตอร์ 1 ตำแหน่ง และตรวจสอบว่าถึงตำแหน่ง Notch หรือไม่

    โฟกัส: การคำนวณการปรับดัชนี (Index Offset) ภายในเมธอด forward() และ backward() เพื่อรองรับ position และ ring_setting เป็นสิ่งสำคัญที่สุดในคลาสนี้

💻 ขั้นตอนที่ 3: รวมส่วนประกอบ (คลาส EnigmaMachine)

คลาสนี้จะจัดการการตั้งค่าเครื่องและการทำงานตามลำดับขั้นตอน
3.1. การสร้างและตั้งค่า (Initialization)

    __init__: รับค่าการตั้งค่าเริ่มต้น เช่น โรเตอร์ที่ใช้ (I, II, III), ตำแหน่งเริ่มต้น (Key Setting), Ring Setting, Reflector ที่ใช้, และการตั้งค่า Plugboard

    จัดเก็บส่วนประกอบย่อย (Plugboard, Rotor Objects 3 ตัว, Reflector) ไว้ใน Attributes

3.2. กลไกการหมุน (_rotate_rotors())

    นี่คือส่วนสำคัญที่ต้องจำลอง Triple-Stepping Mechanism (กลไกการก้าวสามครั้ง)

        โรเตอร์ขวาสุด (III) หมุนทุกครั้ง

        ถ้าโรเตอร์ III หมุนผ่าน Notch ของตัวเอง → โรเตอร์ II หมุน

        ถ้าโรเตอร์ II อยู่ที่ Notch ก่อนการหมุน (Double-Stepping Rule) → โรเตอร์ I และ II จะหมุน

    เรียกเมธอดนี้ ก่อน การเข้ารหัสตัวอักษรทุกครั้ง

3.3. เมธอดเข้ารหัสหลัก (encode_char(char))

    เรียก _rotate_rotors()

    สัญญาณผ่าน Plugboard

    สัญญาณผ่าน Rotors ไปข้างหน้า: Rotor III → Rotor II → Rotor I

    สัญญาณผ่าน Reflector

    สัญญาณผ่าน Rotors ย้อนกลับ: Rotor I → Rotor II → Rotor III

    สัญญาณผ่าน Plugboard ออก

    คืนค่าตัวอักษรที่เข้ารหัส

🧪 ขั้นตอนที่ 4: การทดสอบ

การทดสอบคือสิ่งสำคัญที่สุดในการจำลอง Enigma

    Test 1 (Reciprocity): ลองเข้ารหัสตัวอักษร A ได้ B ต้องแน่ใจว่าเมื่อป้อน B เข้าไปอีกครั้ง (ด้วยการตั้งค่าโรเตอร์เดียวกัน) จะได้ A กลับมา

    Test 2 (Online Simulators): ใช้เว็บไซต์จำลอง Enigma อื่น ๆ เพื่อตั้งค่า (Rotor type, Key, Ring, Plugboard) เหมือนที่คุณตั้ง แล้วทดสอบข้อความสั้น ๆ เพื่อเปรียบเทียบผลลัพธ์ว่าตรงกันหรือไม่

ผมแนะนำให้คุณเริ่มจาก ขั้นตอนที่ 1 และ 2.1 (Reflector) ก่อน เพื่อสร้างรากฐานที่มั่นคงครับ

## Reflector Wirings

In [ ]:
class Reflector():
    
    REFLECTOR_WIRING = {
        'B': 'YRUHQSLDPXNGOKMIEBFZCWVJAT',
        'C': 'FVPJIAOYEDRZXWGCTKUQSBNMHL'
    }
    
    def __init__(self, reflector_type):
        if reflector_type.upper() not in self.REFLECTOR_WIRING:
            raise ValueError(f"Reflector type '{reflector_type}' is invalid.")
        
        self.alphabet = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
        self.wiring = self.REFLECTOR_WIRING[reflector_type.upper()]
        
    def reflect(self, input_index):
        char_out = self.wiring[input_index]
        
        output_index = self.alphabet.index(char_out)
        return output_index

In [ ]:
class Plugboard():
    def __init__(self, pairs):
        self.mapping = {}
        self.alphabet = 'ABCDEFGHIJKLMNOPQRSTUVWXYZ'
    
        for char in self.alphabet:
            self.mapping[char] = char
            
        # input: "AB CD EF"
        pair_list = pairs.upper().split()
        
        for pair in pair_list:
            if len(pair) == 2:
                char1, char2 = pair[0], pair[1]
                
                self.mapping[char1] = char2
                self.mapping[char2] = char1
    
    def swap(self, char):
       if char not in self.mapping:
           return char 
       return self.mapping[char]

In [ ]:
class Rotor:
    ROTOR_DATA = {
        'I':   {'wiring': "EKMFLGDQVZNTOWYHXUSPAIBRCJ", 'notch': 'Q'},
        'II':  {'wiring': "AJDKSIRUXBLHWTMCQGZNPYFVOE", 'notch': 'E'},
        'III': {'wiring': "BDFHJLCPRTXVZNYEIWGAKMUSQO", 'notch': 'V'}
    }

    def __init__(self, name, position='A', ring_setting='A'):
        if name not in self.ROTOR_DATA:
            raise ValueError(f"Unknown rotor: {name}")
            
        data = self.ROTOR_DATA[name]
        
        self.wiring = data['wiring']
        self.notch = data['notch']
        self.alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

In [ ]:
r1 = Rotor('I', position='A')